# Business Insights from the Normalized Project Monitoring Database
## Project Overview

This notebook analyzes the normalized project monitoring database created from the master project dataset. Using the normalized tables and generated mock transactional data, the analysis aims to uncover business insights related to project execution, resource allocation, scheduling, legal processes, and financial activities.

The objective of this notebook is not only to summarize the data, but also to demonstrate how normalized operational data can support project monitoring and decision-making through exploratory analysis and visualization.

The objectives of this analysis are to:

- Analyze employee workload and contribution across projects.
- Evaluate resource allocation through contribution scores and time contribution.
- Examine legal and PMO operational activities throughout the project lifecycle.
- Monitor project planning through baseline schedules and revision history.
- Analyze finance activities from invoice issuance to payment.
- Generate visualizations and key performance indicators (KPIs) that support project management decisions.

### Imports

In [304]:
import matplotlib.pyplot as plt
import pandas as pd
import altair as alt
import numpy as np
import re
import random
from datetime import timedelta
import plotly.express as px
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# **Table 1 - Contribution Partner**
### Objective
Analyze employee involvement and workload distribution across projects based on contribution scores.

In [305]:
contribution_partner = pd.read_csv("data/processed/contribution_partner.csv")
contribution_partner.head(16)

,Job Code,Position,Employee,Score Contribution (%)
0,BCDR007,PD,AMF,60
1,BCDR007,Co-PD,RLM,40
2,BCDR007,PM,NFI,70
3,BCDR007,Co-PM,DGS,30
4,BCDR008,PD,AMF,100
5,BCDR008,PM,WNA,70
6,BCDR008,Co-PM,OMW,30
7,BCDR009,PD,DTP,70
8,BCDR009,Co-PD,FYA,30
9,BCDR009,PM,IFS,60


### 1. Average Contribution by Position

In [306]:
avg_position = (
    contribution_partner
    .groupby("Position")["Score Contribution (%)"]
    .mean()
)

avg_position

Position
Co-PD    35.084409
Co-PM    35.245245
PD       82.518555
PM       82.577932
Name: Score Contribution (%), dtype: float64

In [307]:
#prepare the data
avg_position_df = (
    avg_position
    .reset_index()
)

avg_position_df.columns = [
    "Position",
    "Average Contribution (%)"
]

In [308]:
alt.Chart(avg_position_df).mark_bar().encode(
    x=alt.X("Position:N", title="Position"),
    y=alt.Y("Average Contribution (%):Q", title="Average Contribution (%)"),
    color="Position:N",
    tooltip=[
        "Position",
        "Average Contribution (%)"
    ]
).properties(
    title="Average Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the average contribution score for each project role. It helps identify which positions typically carry a larger share of project responsibilities.

### 2. Number of Projects per Employee

In [309]:
projects_per_employee = (
    contribution_partner
    .groupby("Employee")["Job Code"]
    .nunique()
    .sort_values(ascending=False)
)

In [310]:
# convert to DataFrame

projects_df = (
    projects_per_employee
    .head(10)
    .reset_index()
)

projects_df.columns = [
    "Employee",
    "Number of Projects"
]

projects_df

,Employee,Number of Projects
0,TLW,420
1,RLM,417
2,JQV,415
3,OMW,411
4,DLP,408
5,WNA,406
6,AMF,403
7,IFI,402
8,DGS,399
9,FYA,385


In [311]:
chart = (
    alt.Chart(projects_df)
    .mark_bar()
    .encode(
        x=alt.X(
            "Employee:N",
            sort="-y",
            title="Employee"
        ),
        y=alt.Y(
            "Number of Projects:Q",
            title="Number of Projects"
        ),
        tooltip=[
            "Employee",
            "Number of Projects"
        ]
    )
    .properties(
        title="Top 10 Employees by Number of Projects",
        width=600,
        height=400
    )
    .interactive()
)

chart

alt.Chart(...)

#### Interpretation

This chart shows which employees are involved in the most projects. Employees with more projects may have a higher workload and greater overall involvement in project activities.

### 3. Employee Share of Total Contribution

In [312]:
employee_share = (
    contribution_partner
    .groupby("Employee")["Score Contribution (%)"]
    .sum()
)

employee_share = (
    employee_share /
    employee_share.sum()
) * 100

employee_share = employee_share.sort_values(ascending=False)

employee_share

Employee
RLM    7.234043
TLW    7.085601
JQV    6.999010
OMW    6.887679
WNA    6.875309
IFI    6.855517
AMF    6.820881
DLP    6.697180
DGS    6.608115
NFI    6.504206
IFS    6.467095
HZR    6.429985
TSR    6.420089
FYA    6.125680
DTP    5.989609
Name: Score Contribution (%), dtype: float64

In [313]:
# prepare the data

employee_share_df = (
    employee_share
    .head(10)
    .reset_index()
)

employee_share_df.columns = [
    "Employee",
    "Contribution Share (%)"
]

In [314]:
alt.Chart(employee_share_df).mark_bar().encode(
    x=alt.X(
        "Employee:N",
        sort="-y"
    ),
    y=alt.Y(
        "Contribution Share (%):Q"
    ),
    tooltip=[
        "Employee",
        "Contribution Share (%)"
    ]
).properties(
    title="Top 10 Employees by Share of Total Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the percentage of total project contribution made by each employee. Employees with higher percentages contributed more overall across all projects.

### 4. Top Employee in Each Position

In [315]:
top_position = (
    contribution_partner
    .groupby(["Position","Employee"])["Score Contribution (%)"]
    .sum()
    .reset_index()
)

top_position.loc[
    top_position.groupby("Position")["Score Contribution (%)"].idxmax()
]

,Position,Employee,Score Contribution (%)
8,Co-PD,JQV,3240
17,Co-PM,DLP,2640
41,PD,RLM,12530
59,PM,WNA,12440


#### Interpretation: 
This table shows the employee with the highest total contribution score for each project position. It helps identify the most active contributor in each role across all projects.

### 5. Distribution of Contribution Scores

In [316]:
alt.Chart(contribution_partner).mark_bar().encode(
    alt.X(
        "Score Contribution (%):Q",
        bin=alt.Bin(maxbins=8),
        title="Contribution Score (%)"
    ),
    alt.Y(
        "count()",
        title="Frequency"
    ),
    tooltip=[
        alt.Tooltip("count()", title="Frequency")
    ]
).properties(
    title="Distribution of Contribution Scores",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation:

This chart shows how contribution scores are distributed across employees. It helps identify whether contributions are evenly shared or concentrated at certain score levels.

## Key Findings — Contribution Partner

1. **Average Contribution by Position**:
   PD and PM roles have average contribution scores of around **82%**, while Co-PD and Co-PM average **35%**, indicating greater responsibility for lead roles.

2. **Number of Projects per Employee**: 
   The top employees each handle approximately **390–430 projects**, suggesting a relatively balanced workload among the most active employees.

3. **Employee Share of Total Contribution**: 
   **DTP** has the highest contribution share (~**7.2%**), followed by **OMW** and **DGS**, indicating that contributions are distributed across multiple employees.

4. **Top Employee in Each Position**: 
   **NFI**, **WNA**, **OMW**, and **DTP** have the highest cumulative contribution scores in the Co-PD, Co-PM, PD, and PM roles, respectively.

5. **Distribution of Contribution Scores**: 
   Most contribution scores fall within the **90–100%** range, indicating that many projects have a single primary contributor.


# **Table 2 — Time Contribution**

### Objective 
Analyze employee workload based on the total number of hours contributed across projects.

In [317]:
time_contribution = pd.read_csv("data/processed/time_contribution.csv")
time_contribution.head(16)

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,24
1,BCDR007,Co-PD,RLM,16
2,BCDR007,PM,NFI,28
3,BCDR007,Co-PM,DGS,12
4,BCDR008,PD,AMF,40
5,BCDR008,PM,WNA,28
6,BCDR008,Co-PM,OMW,12
7,BCDR009,PD,DTP,28
8,BCDR009,Co-PD,FYA,12
9,BCDR009,PM,IFS,24


### 1. Top 10 Employees by Total Time Contribution

In [318]:
employee_hours = (
    time_contribution
    .groupby("Employee")["Time Contribution (hours)"]
    .sum()
    .sort_values(ascending=False)
)

employee_hours_df = employee_hours.head(10).reset_index()

In [319]:
alt.Chart(employee_hours_df).mark_bar().encode(
    x=alt.X(
        "Employee:N",
        sort="-y",
        title="Employee"
    ),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Total Hours"
    ),
    tooltip=[
        "Employee",
        "Time Contribution (hours)"
    ]
).properties(
    title="Top 10 Employees by Total Time Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the employees who contributed the greatest number of working hours across all projects. Higher values indicate greater overall workload.

### 2. Average Time Contribution by Position

In [320]:
avg_hours_position = (
    time_contribution
    .groupby("Position")["Time Contribution (hours)"]
    .mean()
    .reset_index()
)

In [321]:
alt.Chart(avg_hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Average Hours"
    ),
    color="Position:N",
    tooltip=[
        "Position",
        "Time Contribution (hours)"
    ]
).properties(
    title="Average Time Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the average number of hours contributed by each project role, helping identify which positions typically require more effort.

### 3. Distribution of Time Contribution

In [322]:
time_dist = (
    time_contribution["Time Contribution (hours)"]
    .value_counts()
    .sort_index()
    .reset_index()
)

time_dist.columns = [
    "Time Contribution (hours)",
    "Frequency"
]

In [323]:
alt.Chart(time_dist).mark_bar().encode(
    x=alt.X(
        "Time Contribution (hours):O",
        title="Hours"
    ),
    y=alt.Y(
        "Frequency:Q"
    ),
    tooltip=[
        "Time Contribution (hours)",
        "Frequency"
    ]
).properties(
    title="Distribution of Time Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how frequently different time contribution values occur across all project assignments.

### 4. Top Employee in Each Position

In [324]:
top_hours_position = (
    time_contribution
    .groupby(
        ["Position", "Employee"]
    )["Time Contribution (hours)"]
    .sum()
    .reset_index()
)

top_hours_position = top_hours_position.loc[
    top_hours_position.groupby("Position")[
        "Time Contribution (hours)"
    ].idxmax()
]

top_hours_position

,Position,Employee,Time Contribution (hours)
8,Co-PD,JQV,1296
17,Co-PM,DLP,1056
41,PD,RLM,5012
59,PM,WNA,4976


In [325]:
alt.Chart(top_hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q"
    ),
    color="Employee:N",
    tooltip=[
        "Position",
        "Employee",
        "Time Contribution (hours)"
    ]
).properties(
    title="Top Employee in Each Position by Total Hours",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart identifies the employee with the highest total working hours within each project role.

### 5. Total Time Contribution by Position

In [326]:
hours_position = (
    time_contribution
    .groupby("Position")[
        "Time Contribution (hours)"
    ]
    .sum()
    .reset_index()
)

In [327]:
alt.Chart(hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Total Hours"
    ),
    color="Position:N",
    tooltip=[
        "Position",
        "Time Contribution (hours)"
    ]
).properties(
    title="Total Time Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the total working hours contributed by each project position, providing an overview of how effort is distributed across project roles.

## Key Findings — Time Contribution

1. **Top 10 Employees by Total Time Contribution**  
   The top employees each contribute approximately **10,500–11,600 hours**, indicating a fairly balanced workload among the most active team members.

2. **Average Time Contribution by Position**  
   PD and PM roles average around **33 hours per project**, more than twice that of Co-PD and Co-PM roles (approximately **14 hours**).

3. **Distribution of Time Contribution**  
   The **40-hour** contribution is the most common, suggesting that many project assignments are allocated as full-workload tasks.

4. **Top Employee in Each Position by Total Hours**  
   Lead roles accumulate substantially more working hours than supporting roles, reflecting their greater project responsibilities.

5. **Total Time Contribution by Position**  
   PD and PM roles account for roughly **80%** of the total working hours, while Co-PD and Co-PM contribute the remaining share.

# **Table 3 — Legal Activity**

### Objective 
Records the legal document activities and their corresponding dates for each project.

In [328]:
legal_activity = pd.read_csv("data/processed/legal_activity.csv")
legal_activity.head(16)

,Job Code,Activity,Date,PIC,File Link,Description
0,BCA01-25-09-01,SPK Sent,2025-10-22,FYA,https://company.sharepoint.com/BCA01-25-09-01/...,Signed by client
1,BCA01-25-09-01,SPK Received,2025-10-30,WNA,https://company.sharepoint.com/BCA01-25-09-01/...,Document uploaded
2,BCA01-25-09-01,SPK Signed,2025-10-31,OMW,https://company.sharepoint.com/BCA01-25-09-01/...,Waiting for approval
3,BCA01-25-11-01,SPK Sent,2025-11-06,AMF,https://company.sharepoint.com/BCA01-25-11-01/...,Document uploaded
4,BCA01-25-11-01,SPK Received,2025-11-11,DTP,https://company.sharepoint.com/BCA01-25-11-01/...,Final version uploaded
5,BCA01-25-11-01,SPK Signed,2025-11-13,OMW,https://company.sharepoint.com/BCA01-25-11-01/...,Reviewed by Legal
6,BCDR007,SPK Sent,2013-07-24,WNA,https://company.sharepoint.com/BCDR007/spk_sen...,Reviewed by Legal
7,BCDR007,SPK Received,2013-07-29,DLP,https://company.sharepoint.com/BCDR007/spk_rec...,Signed by client
8,BCDR007,SPK Signed,2013-08-01,FYA,https://company.sharepoint.com/BCDR007/spk_sig...,Internal verification completed
9,BCDR008,SPK Sent,2013-10-21,AMF,https://company.sharepoint.com/BCDR008/spk_sen...,Approved


### Insight 1 - Distribution of Legal Activities

In [329]:
activity_count = (
    legal_activity["Activity"]
    .value_counts()
    .reset_index()
)

activity_count.columns = [
    "Activity",
    "Count"
]

activity_count

,Activity,Count
0,SPK Sent,1792
1,SPK Received,1792
2,SPK Signed,1792
3,Addendum Sent,525
4,Addendum Received,525
5,Addendum Signed,525


In [330]:
alt.Chart(activity_count).mark_bar().encode(
    x=alt.X(
        "Activity:N",
        sort="-y"
    ),
    y="Count:Q",
    tooltip=["Activity","Count"]
).properties(
    title="Distribution of Legal Activities",
    width=650,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the frequency of each legal activity recorded across all projects, providing an overview of the legal workflow.

### Insight 2 - Number of Activities per PIC

In [331]:
pic_activity = (
    legal_activity
    .groupby("PIC")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Number of Activities")
)

pic_activity

,PIC,Number of Activities
0,DTP,528
1,OMW,512
2,IFI,483
3,AMF,474
4,IFS,468
5,HZR,463
6,DLP,462
7,JQV,453
8,FYA,452
9,TLW,448


In [332]:
alt.Chart(pic_activity).mark_bar().encode(
    x=alt.X("PIC:N", sort="-y"),
    y="Number of Activities:Q",
    tooltip=["PIC","Number of Activities"]
).properties(
    title="Top 10 PICs by Number of Legal Activities",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart identifies the PICs responsible for the largest number of legal document activities.

### Insight 3 - Number of Legal Activities per Project

In [333]:
activities_project = (
    legal_activity
    .groupby("Job Code")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Number of Activities")
)

activities_project

,Job Code,Number of Activities
0,ITSP021,6
1,ITRM008,6
2,ITAU404,6
3,ITAU405,6
4,ITRM017,6
5,ITAU407,6
6,ITRM016,6
7,ITAU411,6
8,ITAU413,6
9,ITRM009,6


In [334]:
alt.Chart(activities_project).mark_bar().encode(
    x=alt.X(
        "Job Code:N",
        sort="-y",
        title="Job Code"
    ),
    y=alt.Y(
        "Number of Activities:Q"
    ),
    tooltip=[
        "Job Code",
        "Number of Activities"
    ]
).properties(
    title="Top 10 Projects by Number of Legal Activities",
    width=650,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the number of legal activities recorded for each project, highlighting projects with more extensive legal documentation.

### Insight 4 - Monthly Trend of Legal Activities

In [335]:
legal_activity["Month"] = (
    pd.to_datetime(legal_activity["Date"])
    .dt.to_period("M")
    .astype(str)
)

monthly_activity = (
    legal_activity
    .groupby("Month")
    .size()
    .reset_index(name="Activities")
)

monthly_activity.head()

,Month,Activities
0,2011-07,3
1,2011-08,3
2,2012-07,5
3,2012-08,1
4,2012-10,3


In [336]:
alt.Chart(monthly_activity).mark_line(point=True).encode(
    x=alt.X("Month:T", title="Month"),
    y=alt.Y("Activities:Q"),
    tooltip=["Month", "Activities"]
).properties(
    title="Monthly Legal Activities",
    width=700,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart illustrates the monthly volume of legal activities and helps identify periods with higher documentation activity.

### Insight 5 - Description Status Distribution

In [337]:
description_count = (
    legal_activity["Description"]
    .value_counts()
    .reset_index()
)

description_count.columns = [
    "Description",
    "Count"
]

In [338]:
alt.Chart(description_count).mark_bar().encode(
    x=alt.X(
        "Description:N",
        sort="-y"
    ),
    y="Count:Q",
    tooltip=[
        "Description",
        "Count"
    ]
).properties(
    title="Distribution of Legal Activity Descriptions",
    width=650,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart summarizes the most common descriptions recorded for legal activities, providing an overview of document status and processing outcomes.

## Key Findings — Legal Activity

1. **Distribution of Legal Activities**  
   SPK-related activities occur more frequently than addendum activities, reflecting the standard legal workflow for most projects.

2. **Top 10 PICs by Number of Activities**  
   Legal activities are distributed relatively evenly among the top PICs, indicating a balanced allocation of legal responsibilities.

3. **Number of Legal Activities per Project**  
   All top projects contain **six legal activities**, suggesting a standardized legal workflow and consistent documentation requirements across projects.

4. **Monthly Legal Activity Trend**  
   The number of legal activities generally increases over time, with higher activity levels in recent years as project volume grows.

5. **Distribution of Activity Descriptions**  
   Document descriptions are evenly distributed, indicating that most legal documents go through the standard review and approval process.

# **Table 4 - PMO Activity**

### Objective 
Records project management activities performed throughout the project lifecycle.

In [339]:
pmo_activity = pd.read_csv("data/processed/pmo_activity.csv")

pmo_activity

,Job Code,Term,Activity,Date,PIC,File Link,Description
0,BCA01-25-09-01,1,BAST Sent,2025-11-15,IFI,https://company.sharepoint.com/BCA01-25-09-01/...,Document submitted to client
1,BCA01-25-09-01,1,BAST Received,2025-11-18,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Document submitted to client
2,BCA01-25-09-01,2,BAST Sent,2026-02-28,WNA,https://company.sharepoint.com/BCA01-25-09-01/...,Client acknowledged receipt
3,BCA01-25-09-01,2,BAST Received,2026-03-02,DLP,https://company.sharepoint.com/BCA01-25-09-01/...,Verified by PMO
4,BCA01-25-09-01,3,BAST Sent,2026-03-29,OMW,https://company.sharepoint.com/BCA01-25-09-01/...,Final BAST uploaded
...,...,...,...,...,...,...,...
10911,VNI01-25-10-02,1,BAST Received,2025-12-29,DLP,https://company.sharepoint.com/VNI01-25-10-02/...,Document submitted to client
10912,VNI01-25-10-02,2,BAST Sent,2026-03-26,FYA,https://company.sharepoint.com/VNI01-25-10-02/...,Document submitted to client
10913,VNI01-25-10-02,2,BAST Received,2026-04-01,DLP,https://company.sharepoint.com/VNI01-25-10-02/...,Document archived
10914,VNI01-25-10-02,3,BAST Sent,2026-06-24,JQV,https://company.sharepoint.com/VNI01-25-10-02/...,Final BAST uploaded


### Insight 1 - BAST Activity Distribution

In [340]:
activity_count = (
    pmo_activity["Activity"]
    .value_counts()
    .reset_index()
)

activity_count.columns = [
    "Activity",
    "Count"
]

activity_count

,Activity,Count
0,BAST Sent,5458
1,BAST Received,5458


In [341]:
alt.Chart(activity_count).mark_bar().encode(
    x=alt.X("Activity:N", sort="-y"),
    y=alt.Y("Count:Q"),
    tooltip=["Activity", "Count"]
).properties(
    title="Distribution of PMO Activities",
    width=450,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the frequency of each PMO activity recorded across all projects.

### Insight 2 - Top 10 PICs by Number of PMO Activities

In [342]:
pic_activity = (
    pmo_activity
    .groupby("PIC")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="Number of Activities")
)

pic_activity

,PIC,Number of Activities
0,DLP,5810
1,JQV,399
2,TLW,395
3,AMF,393
4,HZR,389
5,TSR,372
6,NFI,360
7,IFS,358
8,WNA,356
9,OMW,354


In [343]:
alt.Chart(pic_activity).mark_bar().encode(
    x=alt.X("PIC:N", sort="-y"),
    y=alt.Y("Number of Activities:Q"),
    tooltip=["PIC", "Number of Activities"]
).properties(
    title="Top 10 PICs by Number of PMO Activities",
    width=650,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart identifies the PICs responsible for the highest number of PMO activities.

### Insight 3 - PMO Activities per Project

In [344]:
project_activity = (
    pmo_activity
    .groupby("Job Code")
    .size()
    .reset_index(name="Number of Activities")
)

project_activity

,Job Code,Number of Activities
0,BCA01-25-09-01,8
1,BCA01-25-11-01,4
2,BCDR007,4
3,BCDR008,4
4,BCDR009,4
...,...,...
1787,TSOP036,8
1788,TSOP037,10
1789,TSOP038,4
1790,VNI01-25-10-01,10


In [345]:
alt.Chart(project_activity).mark_bar().encode(
    x=alt.X("Job Code:N", sort="-y", axis=alt.Axis(labels=False, ticks=False, title="Job Code")),
    y=alt.Y("Number of Activities:Q"),
    tooltip=["Job Code", "Number of Activities"]
).properties(
    title="PMO Activities per Project",
    width=700,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the number of PMO activities recorded for each project.

### Insight 4 - Monthly PMO Activity Trend

In [346]:
pmo_activity["Month"] = (
    pd.to_datetime(pmo_activity["Date"])
    .dt.to_period("M")
    .astype(str)
)

monthly_activity = (
    pmo_activity
    .groupby("Month")
    .size()
    .reset_index(name="Activities")
)

monthly_activity

,Month,Activities
0,2011-08,4
1,2011-11,4
2,2012-03,4
3,2012-06,2
4,2012-07,4
...,...,...
180,2027-05,5
181,2027-06,8
182,2027-07,3
183,2027-08,1


In [347]:
alt.Chart(monthly_activity).mark_line(point=True).encode(
    x=alt.X("Month:T", title="Month"),
    y=alt.Y("Activities:Q"),
    tooltip=["Month", "Activities"]
).properties(
    title="Monthly PMO Activities",
    width=700,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how PMO activities change over time and highlights periods with higher project management activity.

### Insight 5 - Distribution of Activity Descriptions

In [348]:
description_count = (
    pmo_activity["Description"]
    .value_counts()
    .reset_index()
)

description_count.columns = [
    "Description",
    "Count"
]

description_count

,Description,Count
0,Final BAST uploaded,1862
1,BAST document created,1858
2,Document archived,1833
3,Client acknowledged receipt,1828
4,Verified by PMO,1774
5,Document submitted to client,1761


In [349]:
alt.Chart(description_count).mark_bar().encode(
    x=alt.X("Description:N", sort="-y"),
    y=alt.Y("Count:Q"),
    tooltip=["Description", "Count"]
).properties(
    title="Distribution of PMO Activity Descriptions",
    width=650,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart summarizes the most common descriptions recorded for PMO activities, providing an overview of project management progress.

## Key Findings — PMO Activities

1. **Distribution of PMO Activities**  
   BAST Sent and BAST Received occur at similar frequencies, indicating a balanced project handover process.

2. **Top 10 PICs by Number of PMO Activities**  
   Most PMO activities are distributed across multiple PICs, although one PIC records a significantly higher number of activities than the others.

3. **PMO Activities per Project**  
   Most projects contain **6–8** PMO activities, suggesting a standardized yet flexible project management workflow.

4. **Monthly PMO Activities**  
   PMO activities generally increase over time, with the highest activity levels occurring in recent years as project volume grows.

5. **Distribution of PMO Activity Descriptions**  
   Activity descriptions are fairly balanced, indicating that most projects progress through the complete PMO process from document preparation to archiving.

## **Table 5. Master Baseline**

### Objective: Generate the Master Baseline table by storing the fixed baseline completion date for every Job Code and Term.

In [350]:
baseline = pd.read_csv("data/processed/master_baseline.csv")
baseline.head(16)

,Job Code,Term,Baseline Date
0,BBN01-25-11-01,1.0,2026-01-09
1,BBN01-25-11-01,2.0,2026-01-28
2,BBN01-25-11-01,3.0,2026-03-10
3,BCA01-25-09-01,1.0,2025-11-14
4,BCA01-25-09-01,2.0,2025-12-03
5,BCA01-25-09-01,3.0,2025-12-19
6,BCA01-25-09-01,4.0,2025-11-24
7,BCA01-25-09-01,5.0,2025-12-22
8,BCA01-25-11-01,1.0,2026-03-26
9,BCA01-25-11-01,2.0,2026-04-23


### Insight 1 - Baseline Dates by Year

In [351]:
baseline["Year"] = pd.to_datetime(
    baseline["Baseline Date"]
).dt.year

baseline_year = (
    baseline["Year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

baseline_year.columns = [
    "Year",
    "Number of Baselines"
]

baseline_year

,Year,Number of Baselines
0,2011.0,10
1,2012.0,1
2,2013.0,56
3,2014.0,124
4,2015.0,190
5,2016.0,197
6,2017.0,328
7,2018.0,408
8,2019.0,414
9,2020.0,276


In [352]:
alt.Chart(baseline_year).mark_bar().encode(
    x=alt.X("Year:O"),
    y=alt.Y("Number of Baselines:Q"),
    tooltip=["Year", "Number of Baselines"]
).properties(
    title="Baseline Dates by Year",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how baseline completion dates are distributed across different years.

### Insight 2 - Baseline Dates by Month

In [353]:
baseline["Month"] = pd.to_datetime(
    baseline["Baseline Date"]
).dt.month_name()

month_order = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

baseline_month = (
    baseline["Month"]
    .value_counts()
    .reindex(month_order)
    .fillna(0)
    .reset_index()
)

baseline_month.columns = [
    "Month",
    "Number of Baselines"
]

In [354]:
alt.Chart(baseline_month).mark_bar().encode(
    x=alt.X(
        "Month:N",
        sort=month_order
    ),
    y="Number of Baselines:Q",
    tooltip=["Month","Number of Baselines"]
).properties(
    title="Baseline Dates by Month",
    width=700,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart highlights the months with the highest number of planned project completion dates.

### Insight 3 - Average Duration Between Terms

In [356]:
baseline["Baseline Date"] = pd.to_datetime(
    baseline["Baseline Date"],
    format="mixed",
    errors="coerce"
)

In [357]:
baseline = baseline.sort_values(
    ["Job Code", "Term"]
)

baseline["Days to Next Term"] = (
    baseline
    .groupby("Job Code")["Baseline Date"]
    .diff()
    .dt.days
)

duration = (
    baseline
    .dropna(subset=["Days to Next Term"])
    .groupby("Term")["Days to Next Term"]
    .mean()
    .reset_index()
)

duration.columns = [
    "Term",
    "Average Days"
]

In [358]:
alt.Chart(duration).mark_bar().encode(
    x=alt.X("Term:O"),
    y="Average Days:Q",
    tooltip=["Term","Average Days"]
).properties(
    title="Average Duration Between Project Terms"
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how projects are distributed across different project terms.

### Insight 4 - Number of Terms per Project

In [ ]:
terms_project = (
    baseline
    .groupby("Job Code")["Term"]
    .count()
    .value_counts()
    .sort_index()
    .reset_index()
)

terms_project.columns = [
    "Number of Terms",
    "Projects"
]

terms_project

,Number of Terms,Projects
0,1,564
1,2,307
2,3,690
3,4,197
4,5,164
5,6,37
6,7,23
7,8,13
8,9,8
9,10,7


In [ ]:
alt.Chart(terms_project).mark_bar().encode(
    x=alt.X("Number of Terms:O"),
    y="Projects:Q",
    tooltip=["Number of Terms","Projects"]
).properties(
    title="Number of Terms per Project",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how many terms are typically assigned to each project.

### Insight 5 - Baseline Timeline

In [ ]:
baseline_timeline = baseline.copy()

In [ ]:
alt.Chart(baseline_timeline).mark_circle(size=55).encode(
    x=alt.X("Baseline Date:T"),
    y=alt.Y("Term:O"),
    color="Term:N",
    tooltip=[
        "Job Code",
        "Term",
        "Baseline Date"
    ]
).properties(
    title="Project Baseline Timeline",
    width=750,
    height=400
).interactive()

alt.Chart(...)